In [4]:
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# CONFIG
# ============================================================

benchmarks = [
    "private_enterprise",
    "social_media_cloud",
    "commercial_cloud",
    "university"
]

# benchmarks = [
#     "private_enterprise",
#     "social_media_cloud"
# ]

loads = range(1, 10)

# Input TrafPy flow files
input_dir = "/home/hsd/workspace/trafpy/examples/comparison_generator/8_hosts_v9(prob1)"

# CONGA / ECMP ns-3 outputs
conga_dir = "/home/hsd/workspace/ns-3.45/conga_sym_8_hosts_v9(prob1)"

# RTT ns-3 outputs
rtt_dir = "/home/hsd/workspace/ns-3.45/rtt_sym_8_hosts_v9(prob1)"

# Output directory
save_dir = "/home/hsd/workspace/ns-3.45/rtt_sym_8_hosts_v9(prob1)/analysis_results"

Path(save_dir).mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# MATCHING CONFIG
# ============================================================

PORT_START = 10000
PORT_END = 50000

ip_pattern = r'^\d+\.\d+\.\d+\.\d+$'


# ============================================================
# FUNCTIONS
# ============================================================

def clean(df):
    """
    Keep only rows that contain valid IP addresses.
    """

    required = [
        "Src",
        "Dest"
    ]

    for col in required:

        if col not in df.columns:

            raise ValueError(
                f"Required column '{col}' not found."
            )

    return df[
        df["Src"].astype(str).str.match(
            ip_pattern,
            na=False
        )
        &
        df["Dest"].astype(str).str.match(
            ip_pattern,
            na=False
        )
    ].copy()


# ============================================================

def ip_to_node(ip):
    """
    Convert ns-3 server IP to input trace node ID.

    Leaf 0:
        10.1.1.2 -> 0
        10.1.1.4 -> 1
        10.1.1.6 -> 2
        10.1.1.8 -> 3

    Leaf 1:
        10.1.2.2 -> 4
        10.1.2.4 -> 5
        10.1.2.6 -> 6
        10.1.2.8 -> 7
    """

    parts = ip.split(".")

    subnet = int(parts[-2])
    host = int(parts[-1])

    base = (host // 2) - 1

    if subnet == 1:

        return base

    elif subnet == 2:

        return base + 4

    else:

        raise ValueError(
            f"Unexpected IP address: {ip}"
        )


# ============================================================

def add_time(df):
    """
    Convert ns-3 TimeFirstTxPacket to seconds.
    """

    df = df.copy()

    df["start_time"] = (
        df["TimeFirstTxPacket"]
        .astype(str)
        .str.replace("+", "", regex=False)
        .str.replace("ns", "", regex=False)
        .astype(float)
        / 1e9
    )

    return df


# ============================================================

def expected_port(flow_index):
    """
    Reproduce the C++ destination-port calculation.

        port = PORT_START + flow.index

    with wrapping after PORT_END.
    """

    port = PORT_START + int(flow_index)

    if port > PORT_END:

        port = port - (
            PORT_END - PORT_START
        )

    return port


# ============================================================
# FLOW COLUMNS
# ============================================================

flow_columns = [
    "FlowID",
    "Src",
    "Dest",
    "TimeFirstRxPacket",
    "TimeFirstTxPacket",
    "TimeLastRxPacket",
    "TimeLastTxPacket",
    "FCT(s)",
    "TxPackets",
    "RxPackets",
    "LostPackets",
    "LossRate",
    "PDR",
    "LossPercent",
    "TxBytes",
    "RxBytes",
    "Throughput(Kbps)",
    "MeanDelay(ms)",
    "Jitter(ms)",
    "HopCount"
]


# ============================================================
# MATCHING FUNCTION
# ============================================================

def match_df(output_df, input_df, label):
    """
    Match input TrafPy flows to ns-3 flows.

    Matching is based on:

        1. source node
        2. destination node
        3. destination port

    If multiple ns-3 flows have the same tuple, the candidate
    with the closest start time to the input event time is used.

    Time is NOT a hard matching criterion.
    """

    matches = []

    # --------------------------------------------------------
    # Make copies
    # --------------------------------------------------------

    output_df = output_df.copy()
    input_df = input_df.copy()

    # --------------------------------------------------------
    # Numeric conversion
    # --------------------------------------------------------

    for col in [
        "SrcPort",
        "DestPort",
        "TxPackets",
        "RxPackets",
        "TxBytes",
        "RxBytes",
        "start_time",
        "sn",
        "dn"
    ]:

        if col in output_df.columns:

            output_df[col] = pd.to_numeric(
                output_df[col],
                errors="coerce"
            )

    for col in [
        "sn",
        "dn",
        "index",
        "flow_size",
        "event_time"
    ]:

        input_df[col] = pd.to_numeric(
            input_df[col],
            errors="coerce"
        )

    # --------------------------------------------------------
    # Remove invalid rows
    # --------------------------------------------------------

    output_df = output_df.dropna(
        subset=[
            "SrcPort",
            "DestPort",
            "TxPackets",
            "RxPackets",
            "TxBytes",
            "RxBytes",
            "start_time",
            "sn",
            "dn"
        ]
    ).copy()

    input_df = input_df.dropna(
        subset=[
            "sn",
            "dn",
            "index",
            "flow_size",
            "event_time"
        ]
    ).copy()

    # --------------------------------------------------------
    # Sort input by event time
    # --------------------------------------------------------

    input_df = input_df.sort_values(
        "event_time"
    ).reset_index(
        drop=True
    )

    # --------------------------------------------------------
    # Create lookup key
    #
    # (source node, destination node, destination port)
    # --------------------------------------------------------

    output_df["_match_key"] = list(
        zip(
            output_df["sn"].astype(int),
            output_df["dn"].astype(int),
            output_df["DestPort"].astype(int)
        )
    )

    # --------------------------------------------------------
    # Group ns-3 flows by matching key
    # --------------------------------------------------------

    grouped = {
        key: group
        for key, group in output_df.groupby(
            "_match_key",
            sort=False
        )
    }

    # --------------------------------------------------------
    # Process every input flow
    # --------------------------------------------------------

    for _, in_row in input_df.iterrows():

        input_flow_id = in_row["flow_id"]

        input_index = int(
            in_row["index"]
        )

        input_src = int(
            in_row["sn"]
        )

        input_dst = int(
            in_row["dn"]
        )

        input_time = float(
            in_row["event_time"]
        )

        input_size = float(
            in_row["flow_size"]
        )

        # ----------------------------------------------------
        # Expected destination port
        # ----------------------------------------------------

        expected_dst_port = expected_port(
            input_index
        )

        # ----------------------------------------------------
        # Lookup
        # ----------------------------------------------------

        key = (
            input_src,
            input_dst,
            expected_dst_port
        )

        candidates = grouped.get(
            key
        )

        # ----------------------------------------------------
        # No matching ns-3 flow
        # ----------------------------------------------------

        if candidates is None or len(candidates) == 0:

            continue

        # ----------------------------------------------------
        # Find closest start time
        # ----------------------------------------------------

        if len(candidates) == 1:

            best = candidates.iloc[0]

            time_diff = abs(
                best["start_time"]
                - input_time
            )

        else:

            time_diffs = (
                candidates["start_time"]
                - input_time
            ).abs()

            best_index = time_diffs.idxmin()

            best = candidates.loc[
                best_index
            ]

            time_diff = time_diffs.loc[
                best_index
            ]

        # ----------------------------------------------------
        # Size difference
        # ----------------------------------------------------

        if input_size > 0:

            size_diff = (
                abs(
                    best["TxBytes"]
                    - input_size
                )
                /
                max(
                    input_size,
                    best["TxBytes"],
                    1
                )
            )

        else:

            size_diff = np.nan

        # ----------------------------------------------------
        # Combined result
        # ----------------------------------------------------

        combined = {

            "flow_id":
                input_flow_id,

            "sn":
                input_src,

            "dn":
                input_dst,

            "input_time":
                input_time,

            "flow_size":
                input_size,

            #"expected_dest_port":
            #    expected_dst_port,

            f"time_diff_{label}":
                time_diff,

            f"size_diff_{label}":
                size_diff,

            f"TxPackets_{label}":
                best["TxPackets"],

            f"RxPackets_{label}":
                best["RxPackets"],

            f"TxBytes_calc_{label}":
                best["TxPackets"] * 1400,

            f"RxBytes_calc_{label}":
                best["RxPackets"] * 1400
        }

        # ----------------------------------------------------
        # Add all FlowMonitor columns
        # ----------------------------------------------------

        for col in flow_columns:

            combined[
                f"{col}_{label}"
            ] = best[col]

        matches.append(
            combined
        )

    return pd.DataFrame(
        matches
    )


# ============================================================
# AVERAGE FCT RESULTS FOR PLOTTING
# ============================================================

avg_fct_results = {
    benchmark: {}
    for benchmark in benchmarks
}



# ============================================================
# MAIN
# ============================================================



for benchmark in benchmarks:

    for load in loads:

        print()
        print(
            "=" * 80
        )

        print(
            f"{benchmark} | Load 0.{load}"
        )

        print(
            "=" * 80
        )

        # ====================================================
        # INPUT TRACE
        # ====================================================

        input_path = (
            f"{input_dir}/"
            f"{benchmark}_load_{load}.csv"
        )

        try:

            input_df = pd.read_csv(
                input_path
            )

            print(
                f"Input flows: "
                f"{len(input_df):,}"
            )

        except Exception as e:

            print(
                f"Could not read input: {e}"
            )

            continue

        # ====================================================
        # ALGORITHMS
        #
        # COMMENT / UNCOMMENT ANYTHING YOU WANT
        # ====================================================

        try:

            algorithms = {

                # ------------------------------------------------
                # CONGA
                # ------------------------------------------------

                "conga": pd.read_csv(
                    f"{conga_dir}/"
                    f"{benchmark}_load_{load}_Conga.csv",
                    nrows=12000
                ),

                # ------------------------------------------------
                # CONGA variants
                # ------------------------------------------------

                # "conga_new": pd.read_csv(
                #     f"{conga_dir}/"
                #     f"{benchmark}_load_{load}_Conga_new.csv",
                #     nrows=12000
                # ),

                # "conga-ecmp": pd.read_csv(
                #     f"{conga_dir}/"
                #     f"{benchmark}_load_{load}_Conga-ECMP_new.csv",
                #     nrows=12000
                # ),

                # ------------------------------------------------
                # ECMP
                # ------------------------------------------------

                "ecmp": pd.read_csv(
                    f"{conga_dir}/"
                    f"{benchmark}_load_{load}_ECMP.csv",
                    nrows=12000
                ),

                # ------------------------------------------------
                # ECMP variants
                # ------------------------------------------------

                # "ecmp_new": pd.read_csv(
                #     f"{conga_dir}/"
                #     f"{benchmark}_load_{load}_ECMP_new.csv",
                #     nrows=12000
                # ),

                # ------------------------------------------------
                # Basic RTT
                # ------------------------------------------------

                # "rtt": pd.read_csv(
                #     f"{rtt_dir}/"
                #     f"{benchmark}_load_{load}_RTT.csv",
                #     nrows=12000
                # ),

                # ------------------------------------------------
                # Weighted ECMP
                # ------------------------------------------------

                # "weighted": pd.read_csv(
                #     f"{rtt_dir}/"
                #     f"{benchmark}_load_{load}_WEIGHTED_ECMP_RTT.csv",
                #     nrows=12000
                # ),

                # ------------------------------------------------
                # Power-of-2 Random
                # ------------------------------------------------

                # "random": pd.read_csv(
                #     f"{rtt_dir}/"
                #     f"{benchmark}_load_{load}_POWER_OF_2_RANDOM_RTT.csv",
                #     nrows=12000
                # ),

                # ------------------------------------------------
                # Power-of-2 Top2
                # ------------------------------------------------

                # "top2": pd.read_csv(
                #     f"{rtt_dir}/"
                #     f"{benchmark}_load_{load}_POWER_OF_2_TOP2_RTT.csv",
                #     nrows=12000
                # ),
            }

            # ====================================================
            # TIMEOUT VARIANTS
            # ====================================================

            timeout_values = [
                "true",
                "false"
            ]

            for timeout in timeout_values:

                # ------------------------------------------------
                # Weighted ECMP + timeout
                # ------------------------------------------------

                algorithms[
                    f"weighted_{timeout}"
                    ] = pd.read_csv(
                    f"{rtt_dir}/"
                    f"{benchmark}_load_{load}_"
                    f"WEIGHTED_ECMP_timeout_{timeout}_RTT.csv",
                    nrows=12000
                )

                # ------------------------------------------------
                # Lowest RTT + timeout
                # ------------------------------------------------

                #algorithms[
                #    f"lrtt_{timeout}"
                #] = pd.read_csv(
                #    f"{rtt_dir}/"
                #    f"{benchmark}_load_{load}_"
                #    f"LOWEST_RTT_timeout_{timeout}_RTT.csv",
                #    nrows=12000
                #)

                # ------------------------------------------------
                # Power-of-2 Random + timeout
                # ------------------------------------------------

                algorithms[
                     f"random_{timeout}"
                     ] = pd.read_csv(
                     f"{rtt_dir}/"
                     f"{benchmark}_load_{load}_"
                     f"POWER_OF_2_RANDOM_timeout_{timeout}_RTT.csv",
                     nrows=12000
                 )

                # ------------------------------------------------
                # Power-of-2 Top2 + timeout
                # ------------------------------------------------

                # algorithms[
                #     f"top2_{timeout}"
                # ] = pd.read_csv(
                #     f"{rtt_dir}/"
                #     f"{benchmark}_load_{load}_"
                #     f"POWER_OF_2_TOP2_timeout_{timeout}_RTT.csv",
                #     nrows=12000
                # )

        except Exception as e:

            print(
                "Skipping:",
                e
            )

            continue

        # ====================================================
        # CLEAN + PREPARE
        # ====================================================

        for label, df in algorithms.items():

            try:

                df = clean(
                    df
                )

                df = add_time(
                    df
                )

                df["sn"] = df["Src"].apply(
                    ip_to_node
                )

                df["dn"] = df["Dest"].apply(
                    ip_to_node
                )

                algorithms[label] = df

                print(
                    f"{label}: "
                    f"{len(df):,} IP flows"
                )

            except Exception as e:

                print(
                    f"ERROR processing "
                    f"{label}: {e}"
                )

                algorithms[label] = (
                    pd.DataFrame()
                )

        # ====================================================
        # MATCH
        # ====================================================

        matched = {}

        for label, df in algorithms.items():

            if df.empty:

                print(
                    f"Skipping matching for "
                    f"{label}: empty dataframe"
                )

                continue

            print(
                f"Matching {label}..."
            )

            matched[label] = match_df(
                df,
                input_df,
                label
            )

            print(
                f"{label}: "
                f"{len(matched[label]):,} matches"
            )

        # ====================================================
        # REMOVE EMPTY ALGORITHMS
        # ====================================================

        matched = {
            label: df
            for label, df in matched.items()
            if not df.empty
        }

        if len(matched) == 0:

            print(
                "No matches."
            )

            continue

        # ====================================================
        # MERGE RESULTS
        # ====================================================

        merge_keys = [
            "flow_id",
            "sn",
            "dn",
            "input_time",
            "flow_size"
        ]

        algorithms_list = list(
            matched.keys()
        )

        final_df = matched[
            algorithms_list[0]
        ]

        for algo in algorithms_list[1:]:

            final_df = pd.merge(
                final_df,
                matched[algo],
                on=merge_keys,
                how="inner"
            )

        # ====================================================
        # CHECK RESULT
        # ====================================================

        if len(final_df) == 0:

            print(
                "No common matches across "
                "the selected algorithms."
            )

            continue

        print()
        print(
            f"Final matched flows: "
            f"{len(final_df):,}"
        )


        # ============================================================
        # MATCHING TIME-DIFFERENCE SANITY CHECK
        # ============================================================
        
        print("\n" + "=" * 80)
        print("MATCHING TIME-DIFFERENCE CHECK")
        print("=" * 80)
        
        for algo in algorithms:
        
            col = f"time_diff_{algo}"
        
            if col not in final_df.columns:
                continue
        
            max_diff = final_df[col].max()
            mean_diff = final_df[col].mean()
            p95_diff = final_df[col].quantile(0.95)
            p99_diff = final_df[col].quantile(0.99)
        
            max_idx = final_df[col].idxmax()
            worst = final_df.loc[max_idx]
        
            print(f"\n{algo}")
            print(f"  Maximum : {max_diff:.9f} s "
                  f"({max_diff * 1000:.6f} ms)")
            print(f"  Mean    : {mean_diff:.9f} s "
                  f"({mean_diff * 1000:.6f} ms)")
            print(f"  P95     : {p95_diff:.9f} s "
                  f"({p95_diff * 1000:.6f} ms)")
            print(f"  P99     : {p99_diff:.9f} s "
                  f"({p99_diff * 1000:.6f} ms)")
        
            print(f"  Worst matching flow:")
            print(f"    flow_id     : {worst['flow_id']}")
            print(f"    sn          : {worst['sn']}")
            print(f"    dn          : {worst['dn']}")
            print(f"    input_time  : {worst['input_time']:.9f} s")
            print(f"    time_diff   : {worst[col]:.9f} s")

        
        # ====================================================
        # DIFFERENCES
        # ====================================================

        reference = "ecmp"

        if reference in matched:

            for algo in algorithms:

                if algo == reference:
                    continue

                if (
                    f"FCT(s)_{reference}" in final_df.columns
                    and
                    f"FCT(s)_{algo}" in final_df.columns
                ):

                    final_df[
                        f"fct_diff_{reference}_{algo}"
                    ] = (
                        final_df[
                            f"FCT(s)_{reference}"
                        ]
                        -
                        final_df[
                            f"FCT(s)_{algo}"
                        ]
                    )

        # ====================================================
        # SAVE
        # ====================================================

        csv_path = (
            f"{save_dir}/"
            f"{benchmark}_load_{load}_full.csv"
        )

        final_df.to_csv(
            csv_path,
            index=False
        )

        print(
            f"Saved: {csv_path}"
        )

        # ====================================================
        # STATS
        # ====================================================

        print()
        print(
            "-" * 80
        )

        print(
            f"Statistics: "
            f"{benchmark} | Load 0.{load}"
        )

        print(
            "-" * 80
        )

        for algo in matched:

            column = (
                f"FCT(s)_{algo}"
            )

            if column not in final_df.columns:
                continue

            fct = final_df[column]

            avg_fct = fct.mean()

            # --------------------------------------------------------
            # Store average FCT for later plotting
            # --------------------------------------------------------
        
            avg_fct_results[benchmark].setdefault(algo,{})
        
            avg_fct_results[benchmark][algo][load] = avg_fct



            print(
                f"{algo:20s}"
                f"Avg: {fct.mean():.6f}  "
                f"P75: {fct.quantile(0.75):.6f}  "
                f"P90: {fct.quantile(0.90):.6f}  "
                f"P95: {fct.quantile(0.95):.6f}  "
                f"P99: {fct.quantile(0.99):.6f}"
            )


# ============================================================
# DONE
# ============================================================

# ============================================================
# AVERAGE FCT COMPARISON GRAPHS
# ============================================================

print()
print("=" * 80)
print("GENERATING AVERAGE FCT COMPARISON GRAPHS")
print("=" * 80)


for benchmark in benchmarks:

    benchmark_results = avg_fct_results.get(
        benchmark,
        {}
    )

    if not benchmark_results:

        print(
            f"No FCT results available for "
            f"{benchmark}"
        )

        continue

    # --------------------------------------------------------
    # Create figure
    # --------------------------------------------------------

    plt.figure(
        figsize=(10, 6)
    )

    # --------------------------------------------------------
    # Plot each algorithm
    # --------------------------------------------------------

    for algo, load_results in benchmark_results.items():

        if not load_results:
            continue

        x = sorted(
            load_results.keys()
        )

        y = [
            load_results[load]
            for load in x
        ]

        # Convert load 1..9 to 0.1..0.9
        x_values = [
            load / 10
            for load in x
        ]

        plt.plot(
            x_values,
            y,
            marker="o",
            linewidth=2,
            markersize=6,
            label=algo
        )

    # --------------------------------------------------------
    # Labels
    # --------------------------------------------------------

    plt.xlabel(
        "Load",
        fontsize=12
    )

    plt.ylabel(
        "Average FCT (s)",
        fontsize=12
    )

    plt.title(
        f"Average Flow Completion Time - {benchmark}",
        fontsize=14
    )

    # --------------------------------------------------------
    # X-axis
    # --------------------------------------------------------

    plt.xticks(
        np.arange(
            0.1,
            1.0,
            0.1
        )
    )

    # --------------------------------------------------------
    # Grid
    # --------------------------------------------------------

    plt.grid(
        True,
        linestyle="--",
        alpha=0.5
    )

    # --------------------------------------------------------
    # Legend
    # --------------------------------------------------------

    plt.legend(
        fontsize=10
    )

    # --------------------------------------------------------
    # Layout
    # --------------------------------------------------------

    plt.tight_layout()

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    # ============================================================
    # GRAPH OUTPUT DIRECTORIES
    # ============================================================
    
    graph_dir = Path(save_dir) / "graph"
    avg_graph_dir = graph_dir / "avg"
    
    avg_graph_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    graph_path = (
        avg_graph_dir /
        f"{benchmark}_average_fct_comparison.png"
    )

    plt.savefig(
        graph_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print(
        f"Saved graph: {graph_path}"
    )

print(
    "MATCHING COMPLETE"
)

print(
    "=" * 80
)


private_enterprise | Load 0.1
Input flows: 6,000
conga: 12,000 IP flows
ecmp: 12,000 IP flows
weighted_true: 12,000 IP flows
random_true: 12,000 IP flows
weighted_false: 12,000 IP flows
random_false: 12,000 IP flows
Matching conga...
conga: 6,000 matches
Matching ecmp...
ecmp: 6,000 matches
Matching weighted_true...
weighted_true: 6,000 matches
Matching random_true...
random_true: 6,000 matches
Matching weighted_false...
weighted_false: 6,000 matches
Matching random_false...
random_false: 6,000 matches

Final matched flows: 6,000

MATCHING TIME-DIFFERENCE CHECK

conga
  Maximum : 0.000000000 s (0.000000 ms)
  Mean    : 0.000000000 s (0.000000 ms)
  P95     : 0.000000000 s (0.000000 ms)
  P99     : 0.000000000 s (0.000000 ms)
  Worst matching flow:
    flow_id     : flow_123
    sn          : 6
    dn          : 1
    input_time  : 1.972790288 s
    time_diff   : 0.000000000 s

ecmp
  Maximum : 0.000000000 s (0.000000 ms)
  Mean    : 0.000000000 s (0.000000 ms)
  P95     : 0.000000000 

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# ------------------ CONFIG ------------------

save_dir = Path(
    "/home/hsd/workspace/ns-3.45/rtt_sym_8_hosts_v9(prob1)/analysis_results"
)

graph_dir = save_dir / "graph"
graph_dir.mkdir(parents=True, exist_ok=True)

benchmarks = [
    "private_enterprise",
    "social_media_cloud",
    "commercial_cloud",
    "university"
]

loads = range(1, 10)

SMALL = 100 * 1024        # 100 KB
LARGE = 1 * 1024 * 1024   # 1 MB

algorithms = [
    "conga",
    #"conga_new",
    #"conga-ecmp",
    "ecmp",
    #"ecmp_new",
    
    #"lrtt_true",
    #"lrtt_false",
    
    #"weighted",
    #"random",
    #"top2"
    
    "weighted_true",
    "weighted_false",

    "random_true",
    "random_false",

    #"top2_true",
    #"top2_false"
]

label_map = {
    "conga": "Conga",
    #"conga_new": "Conga_New",
    #"conga-ecmp": "Conga-ECMP",
    "ecmp": "ECMP",
    #"ecmp_new": "ECMP_New",
    #"rtt": "RTT",

    "lrtt_true": "Lowest RTT (Timeout)",
    "lrtt_false": "Lowest RTT (No Timeout)",
    
    "weighted": "Weighted ECMP",
    "random": "Power-of-2 Random",
    "top2": "Power-of-2 Top2",
    
    "weighted_true": "Weighted ECMP (Timeout)",
    "weighted_false": "Weighted ECMP (No Timeout)",

    "random_true": "Power-of-2 Random (Timeout)",
    "random_false": "Power-of-2 Random (No Timeout)",

    "top2_true": "Power-of-2 Top2 (Timeout)",
    "top2_false": "Power-of-2 Top2 (No Timeout)"
}

color_map = {
    "conga": "tab:blue",
    #"conga_new": "tab:orange",
    #"conga-ecmp": "tab:green",
    "ecmp": "tab:red",
    #"ecmp_new": "tab:purple",
    #"rtt": "tab:brown",

    "lrtt_true": "tab:cyan",
    "lrtt_false": "tab:cyan",
    
    "weighted": "tab:pink",
    "random": "tab:gray",
    "top2": "tab:olive",
    
    "weighted_true": "tab:brown",
    "weighted_false": "tab:brown",

    "random_true": "tab:orange",
    "random_false": "tab:orange",

    "top2_true": "tab:gray",
    "top2_false": "tab:gray",
}

marker_map = {
    "conga": "o",
    #"conga_new": "x",
    #"conga-ecmp": "X",
    "ecmp": "s",
    #"ecmp_new": "*",
    #"rtt": "^",

    "lrtt_true": "1",
    "lrtt_false": "2",
    
    "weighted": "D",
    "random": "v",
    "top2": "P",
    
    "weighted_true": "D",
    "weighted_false": "d",

    "random_true": "^",
    "random_false": "v",

    "top2_true": "P",
    "top2_false": "X"
}

# ------------------ GLOBAL Y-LIMITS ------------------

small_global = []
large_global = []

for benchmark in benchmarks:
    for load in loads:

        csv_path = save_dir / f"{benchmark}_load_{load}_full.csv"

        if not csv_path.exists():
            continue

        df = pd.read_csv(csv_path)

        if df.empty:
            continue

        small_df = df[df["flow_size"] < SMALL]
        large_df = df[df["flow_size"] > LARGE]

        for algo in algorithms:
            col = f"FCT(s)_{algo}"

            if not small_df.empty:
                small_global.append(small_df[col].mean())

            if not large_df.empty:
                large_global.append(large_df[col].mean())

small_min = min(small_global)
small_max = max(small_global)

large_min = min(large_global)
large_max = max(large_global)

small_margin = 0.05 * (small_max - small_min)
large_margin = 0.05 * (large_max - large_min)

small_ylim = (small_min - small_margin, small_max + small_margin)
large_ylim = (large_min - large_margin, large_max + large_margin)

# ------------------ ANALYSIS ------------------

for benchmark in benchmarks:

    print(f"\n===== {benchmark} =====")

    load_vals = []

    small_results = {algo: [] for algo in algorithms}
    large_results = {algo: [] for algo in algorithms}

    for load in loads:

        csv_path = save_dir / f"{benchmark}_load_{load}_full.csv"

        if not csv_path.exists():
            print(f"Missing: {csv_path}")
            continue

        df = pd.read_csv(csv_path)

        if df.empty:
            continue

        small_df = df[df["flow_size"] < SMALL]
        large_df = df[df["flow_size"] > LARGE]

        # ---------- Small Flows ----------

        for algo in algorithms:

            col = f"FCT(s)_{algo}"

            if not small_df.empty:
                small_results[algo].append(
                    small_df[col].mean()
                )
            else:
                small_results[algo].append(np.nan)

        # ---------- Large Flows ----------

        for algo in algorithms:

            col = f"FCT(s)_{algo}"

            if not large_df.empty:
                large_results[algo].append(
                    large_df[col].mean()
                )
            else:
                large_results[algo].append(np.nan)

        load_vals.append(load / 10)

    # ==================================================
    # SMALL FLOWS
    # ==================================================

    plt.figure(figsize=(8, 5))

    for algo in algorithms:

        plt.plot(
            load_vals,
            small_results[algo],
            marker=marker_map[algo],
            color=color_map[algo],
            linestyle="--" if algo.endswith("_false") else "-",
            linewidth=2,
            label=label_map[algo]
        )

    plt.xlabel("Network Load")
    plt.ylabel("Average FCT (s)")
    plt.title(f"{benchmark}: Small Flows (<100 KB)")
    #plt.ylim(small_ylim)
    plt.grid(True)
    plt.legend()

    out_path = graph_dir / f"{benchmark}_SMALL_vs_load.png"

    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()

    print(f"Saved: {out_path}")

    # ==================================================
    # LARGE FLOWS
    # ==================================================

    plt.figure(figsize=(8, 5))

    for algo in algorithms:

        plt.plot(
            load_vals,
            large_results[algo],
            marker=marker_map[algo],
            color=color_map[algo],
            linestyle="--" if algo.endswith("_false") else "-",
            linewidth=2,
            label=label_map[algo]
        )

    plt.xlabel("Network Load")
    plt.ylabel("Average FCT (s)")
    plt.title(f"{benchmark}: Large Flows (>1 MB)")
    #plt.ylim(large_ylim)
    plt.grid(True)
    plt.legend()

    out_path = graph_dir / f"{benchmark}_LARGE_vs_load.png"

    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.close()

    print(f"Saved: {out_path}")

print("\nDone: Small vs Large flow comparison for all algorithms.")


===== private_enterprise =====
Saved: /home/hsd/workspace/ns-3.45/rtt_sym_8_hosts_v9(prob1)/analysis_results/graph/private_enterprise_SMALL_vs_load.png
Saved: /home/hsd/workspace/ns-3.45/rtt_sym_8_hosts_v9(prob1)/analysis_results/graph/private_enterprise_LARGE_vs_load.png

===== social_media_cloud =====
Saved: /home/hsd/workspace/ns-3.45/rtt_sym_8_hosts_v9(prob1)/analysis_results/graph/social_media_cloud_SMALL_vs_load.png
Saved: /home/hsd/workspace/ns-3.45/rtt_sym_8_hosts_v9(prob1)/analysis_results/graph/social_media_cloud_LARGE_vs_load.png

===== commercial_cloud =====
Saved: /home/hsd/workspace/ns-3.45/rtt_sym_8_hosts_v9(prob1)/analysis_results/graph/commercial_cloud_SMALL_vs_load.png
Saved: /home/hsd/workspace/ns-3.45/rtt_sym_8_hosts_v9(prob1)/analysis_results/graph/commercial_cloud_LARGE_vs_load.png

===== university =====
Saved: /home/hsd/workspace/ns-3.45/rtt_sym_8_hosts_v9(prob1)/analysis_results/graph/university_SMALL_vs_load.png
Saved: /home/hsd/workspace/ns-3.45/rtt_sym_8_ho

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np

# ============================================================
# CONFIG
# ============================================================

save_dir = Path(
    "/home/hsd/workspace/ns-3.45/rtt_sym_8_hosts_v9(prob1)/analysis_results"
)

baseline = "ecmp"

graph_dir = save_dir / "graph" / f"normalised_vs_{baseline}"
graph_dir.mkdir(parents=True, exist_ok=True)

benchmarks = [
    "private_enterprise",
    "social_media_cloud",
    "commercial_cloud",
    "university"
]

# ------------------------------------------------------------
# Loads
# ------------------------------------------------------------

loads = range(1, 10)

# ------------------------------------------------------------
# Flow size thresholds
# ------------------------------------------------------------

SMALL = 100 * 1024          # 100 KB
LARGE = 1 * 1024 * 1024     # 1 MB


# ============================================================
# ALGORITHMS
#
# Comment/uncomment algorithms here as required.
# ECMP is the baseline and therefore does NOT need to be here.
# ============================================================

algorithms = [

    "conga",

    # "lrtt_true",
    # "lrtt_false",

    "weighted_true",
    "weighted_false",

    "random_true",
    "random_false",

    # "top2_true",
    # "top2_false",

    # "weighted",
    # "random",
    # "top2",
]


# ============================================================
# LABELS
# ============================================================

label_map = {

    "conga":
        "Conga",

    "lrtt_true":
        "Lowest RTT (Timeout)",

    "lrtt_false":
        "Lowest RTT (No Timeout)",

    "weighted":
        "Weighted ECMP",

    "random":
        "Power-of-2 Random",

    "top2":
        "Power-of-2 Top2",

    "weighted_true":
        "Weighted ECMP (Timeout)",

    "weighted_false":
        "Weighted ECMP (No Timeout)",

    "random_true":
        "Power-of-2 Random (Timeout)",

    "random_false":
        "Power-of-2 Random (No Timeout)",

    "top2_true":
        "Power-of-2 Top2 (Timeout)",

    "top2_false":
        "Power-of-2 Top2 (No Timeout)",
}


# ============================================================
# COLORS
# ============================================================

color_map = {

    "conga":
        "tab:blue",

    "lrtt_true":
        "tab:cyan",

    "lrtt_false":
        "tab:cyan",

    "weighted":
        "tab:purple",

    "random":
        "tab:gray",

    "top2":
        "tab:olive",

    "weighted_true":
        "tab:brown",

    "weighted_false":
        "tab:brown",

    "random_true":
        "tab:orange",

    "random_false":
        "tab:orange",

    "top2_true":
        "tab:gray",

    "top2_false":
        "tab:gray",
}


# ============================================================
# MARKERS
# ============================================================

marker_map = {

    "conga":
        "o",

    "lrtt_true":
        "1",

    "lrtt_false":
        "2",

    "weighted":
        "^",

    "random":
        "D",

    "top2":
        "P",

    "weighted_true":
        "D",

    "weighted_false":
        "d",

    "random_true":
        "^",

    "random_false":
        "v",

    "top2_true":
        "P",

    "top2_false":
        "X",
}


# ============================================================
# RESULT STORAGE
# ============================================================

summary_results = {}


# ============================================================
# MAIN ANALYSIS
# ============================================================

for benchmark in benchmarks:

    print()
    print("=" * 80)
    print(f"BENCHMARK: {benchmark}")
    print("=" * 80)

    # --------------------------------------------------------
    # Store load values
    # --------------------------------------------------------

    load_vals = []

    # --------------------------------------------------------
    # Results for each algorithm
    #
    # Each entry will contain one value per network load.
    # --------------------------------------------------------

    small_results = {
        algo: []
        for algo in algorithms
    }

    large_results = {
        algo: []
        for algo in algorithms
    }

    # ========================================================
    # PROCESS EACH LOAD
    # ========================================================

    for load in loads:

        print()
        print(
            f"Processing {benchmark} | Load 0.{load}"
        )

        csv_path = (
            save_dir /
            f"{benchmark}_load_{load}_full.csv"
        )

        # ----------------------------------------------------
        # Check file
        # ----------------------------------------------------

        if not csv_path.exists():

            print(
                f"Missing: {csv_path}"
            )

            continue

        # ----------------------------------------------------
        # Read CSV
        # ----------------------------------------------------

        df = pd.read_csv(
            csv_path
        )

        if df.empty:

            print(
                "Empty dataframe."
            )

            continue

        # ====================================================
        # CHECK REQUIRED COLUMNS
        # ====================================================

        baseline_col = (
            f"FCT(s)_{baseline}"
        )

        if baseline_col not in df.columns:

            print(
                f"ERROR: Missing baseline column "
                f"{baseline_col}"
            )

            continue

        missing_algorithms = []

        for algo in algorithms:

            col = f"FCT(s)_{algo}"

            if col not in df.columns:

                missing_algorithms.append(
                    col
                )

        if missing_algorithms:

            print(
                "ERROR: Missing algorithm columns:"
            )

            for col in missing_algorithms:

                print(
                    f"    {col}"
                )

            continue

        # ====================================================
        # REMOVE INVALID BASELINE FCT VALUES
        # ====================================================

        df = df[
            df[baseline_col] > 0
        ].copy()

        if df.empty:

            print(
                "No valid flows after removing "
                "zero/negative ECMP FCT."
            )

            continue

        # ====================================================
        # FLOW SIZE SPLIT
        # ====================================================

        small_df = df[
            df["flow_size"] < SMALL
        ].copy()

        large_df = df[
            df["flow_size"] > LARGE
        ].copy()

        print(
            f"Small flows: {len(small_df):,}"
        )

        print(
            f"Large flows: {len(large_df):,}"
        )

        # ====================================================
        # BASELINE AVERAGES
        #
        # IMPORTANT:
        #
        # We calculate:
        #
        #     mean(FCT_algorithm)
        #     --------------------
        #     mean(FCT_ECMP)
        #
        # NOT:
        #
        #     mean(FCT_algorithm / FCT_ECMP)
        #
        # This makes the normalized graph consistent with
        # the Average FCT graph.
        # ====================================================

        # ----------------------------------------------------
        # SMALL FLOWS
        # ----------------------------------------------------

        if not small_df.empty:

            baseline_small_avg = (
                small_df[baseline_col].mean()
            )

        else:

            baseline_small_avg = np.nan

        # ----------------------------------------------------
        # LARGE FLOWS
        # ----------------------------------------------------

        if not large_df.empty:

            baseline_large_avg = (
                large_df[baseline_col].mean()
            )

        else:

            baseline_large_avg = np.nan

        # ====================================================
        # NORMALIZED RESULTS
        # ====================================================

        for algo in algorithms:

            algo_col = (
                f"FCT(s)_{algo}"
            )

            # ------------------------------------------------
            # SMALL FLOWS
            # ------------------------------------------------

            if (
                not small_df.empty
                and baseline_small_avg > 0
            ):

                algo_small_avg = (
                    small_df[algo_col].mean()
                )

                normalized_small = (
                    algo_small_avg /
                    baseline_small_avg
                )

                small_results[algo].append(
                    normalized_small
                )

            else:

                small_results[algo].append(
                    np.nan
                )

            # ------------------------------------------------
            # LARGE FLOWS
            # ------------------------------------------------

            if (
                not large_df.empty
                and baseline_large_avg > 0
            ):

                algo_large_avg = (
                    large_df[algo_col].mean()
                )

                normalized_large = (
                    algo_large_avg /
                    baseline_large_avg
                )

                large_results[algo].append(
                    normalized_large
                )

            else:

                large_results[algo].append(
                    np.nan
                )

        # ----------------------------------------------------
        # Store load
        # ----------------------------------------------------

        load_vals.append(
            load / 10
        )

        # ====================================================
        # PRINT NORMALIZED VALUES
        # ====================================================

        print()
        print(
            "Normalized Average FCT:"
        )

        if not small_df.empty:

            print(
                f"  Small flows "
                f"(ECMP avg = "
                f"{baseline_small_avg:.6f}s)"
            )

            for algo in algorithms:

                value = (
                    small_results[algo][-1]
                )

                print(
                    f"    {label_map[algo]:35s}: "
                    f"{value:.4f}"
                )

        if not large_df.empty:

            print(
                f"  Large flows "
                f"(ECMP avg = "
                f"{baseline_large_avg:.6f}s)"
            )

            for algo in algorithms:

                value = (
                    large_results[algo][-1]
                )

                print(
                    f"    {label_map[algo]:35s}: "
                    f"{value:.4f}"
                )


    # ========================================================
    # STORE RESULTS
    # ========================================================

    summary_results[benchmark] = {

        "loads":
            load_vals,

        "small":
            small_results,

        "large":
            large_results
    }


    # ========================================================
    # SMALL FLOWS PLOT
    # ========================================================

    plt.figure(
        figsize=(10, 7),
        dpi=120
    )

    for algo in algorithms:

        plt.plot(

            load_vals,

            small_results[algo],

            marker=marker_map[algo],

            color=color_map[algo],

            linestyle=(
                "--"
                if algo.endswith("_false")
                else "-"
            ),

            linewidth=2,

            markersize=7,

            label=(
                f"{label_map[algo]} / ECMP"
            )
        )

    # --------------------------------------------------------
    # ECMP baseline
    # --------------------------------------------------------

    plt.axhline(
        y=1,
        linestyle="--",
        color="black",
        linewidth=1.5,
        label="ECMP baseline"
    )

    # --------------------------------------------------------
    # Labels
    # --------------------------------------------------------

    plt.xlabel(
        "Network Load"
    )

    plt.ylabel(
        "Normalized Average FCT"
    )

    plt.title(
        f"{benchmark}: "
        f"Small Flows (<100 KB)"
    )

    # --------------------------------------------------------
    # Y-axis
    # --------------------------------------------------------

    plt.ylim(
        0,
        1.8
    )

    plt.yticks(
        np.arange(
            0,
            1.81,
            0.2
        )
    )

    # --------------------------------------------------------
    # Grid
    # --------------------------------------------------------

    plt.grid(
        True,
        alpha=0.3
    )

    plt.legend()

    plt.tight_layout()

    out_path = (
        graph_dir /
        f"{benchmark}_SMALL_relative_vs_load.png"
    )

    plt.savefig(
        out_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print()
    print(
        f"Saved: {out_path}"
    )


    # ========================================================
    # LARGE FLOWS PLOT
    # ========================================================

    plt.figure(
        figsize=(10, 7),
        dpi=120
    )

    for algo in algorithms:

        plt.plot(

            load_vals,

            large_results[algo],

            marker=marker_map[algo],

            color=color_map[algo],

            linestyle=(
                "--"
                if algo.endswith("_false")
                else "-"
            ),

            linewidth=2,

            markersize=7,

            label=(
                f"{label_map[algo]} / ECMP"
            )
        )

    # --------------------------------------------------------
    # ECMP baseline
    # --------------------------------------------------------

    plt.axhline(
        y=1,
        linestyle="--",
        color="black",
        linewidth=1.5,
        label="ECMP baseline"
    )

    # --------------------------------------------------------
    # Labels
    # --------------------------------------------------------

    plt.xlabel(
        "Network Load"
    )

    plt.ylabel(
        "Normalized Average FCT"
    )

    plt.title(
        f"{benchmark}: "
        f"Large Flows (>1 MB)"
    )

    # --------------------------------------------------------
    # Y-axis
    # --------------------------------------------------------

    plt.ylim(
        0,
        1.4
    )

    plt.yticks(
        np.arange(
            0,
            1.41,
            0.2
        )
    )

    # --------------------------------------------------------
    # Grid
    # --------------------------------------------------------

    plt.grid(
        True,
        alpha=0.3
    )

    plt.legend()

    plt.tight_layout()

    out_path = (
        graph_dir /
        f"{benchmark}_LARGE_relative_vs_load.png"
    )

    plt.savefig(
        out_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()

    print(
        f"Saved: {out_path}"
    )


# ============================================================
# DONE
# ============================================================

print()
print("=" * 80)
print(
    "DONE: Normalized Average FCT graphs generated."
)
print("=" * 80)


BENCHMARK: private_enterprise

Processing private_enterprise | Load 0.1
Small flows: 5,790
Large flows: 26

Normalized Average FCT:
  Small flows (ECMP avg = 0.003808s)
    Conga                              : 0.7321
    Weighted ECMP (Timeout)            : 0.8234
    Weighted ECMP (No Timeout)         : 0.9356
    Power-of-2 Random (Timeout)        : 1.1936
    Power-of-2 Random (No Timeout)     : 1.1483
  Large flows (ECMP avg = 0.296755s)
    Conga                              : 0.8978
    Weighted ECMP (Timeout)            : 0.9234
    Weighted ECMP (No Timeout)         : 0.9209
    Power-of-2 Random (Timeout)        : 1.0903
    Power-of-2 Random (No Timeout)     : 1.0008

Processing private_enterprise | Load 0.2
Small flows: 5,790
Large flows: 26

Normalized Average FCT:
  Small flows (ECMP avg = 0.006633s)
    Conga                              : 0.5857
    Weighted ECMP (Timeout)            : 0.7260
    Weighted ECMP (No Timeout)         : 1.0275
    Power-of-2 Random (Timeout